In [7]:
# load txt, and read the list of the corresponding line in the txt
import os, shutil, sys, re
import numpy as np
DA_list = []
cT_list = []
pT_list = []
a_list = []
version_list = []
cfa_weight_list = []
mAP_list = []
mAR_list = []
mFscore_list = []
cfg_name_list = []
APClassWise_list = []
ARClassWise_list = []
FscoreClassWise_list = []

def read_log_save_csv(log_path, csv_path):
    print(log_path)
    cfg_name = log_path.split("/")
    print(cfg_name)
    cfg_name = cfg_name[-2]
    # read ap ar f
    with open(log_path, "r") as f:
        srcCfg = f.read().split("\n")
        keys = ['AP50', 'AR50']
        result = dict()
        for key in keys:
            result[key] = [0.0] * 5 
        pattern = re.compile(r'\d+\.\d+') # 非负浮点数
        for key in keys:
            for i in range(len(srcCfg)):
                if key in srcCfg[i] and ':' not in srcCfg[i]:
                    index_3 = i+4
                    index_6 = i+5
                    index_9 = i+6
                    index_12 = i+7
                    AP50_list_3 = (pattern.findall(srcCfg[index_3]))
                    AP50_list_6 = list()
                    AP50_list_9 = list()
                    AP50_list_12 = list()
                    if AP50_list_3 != []:
                        AP50_list_6 = (pattern.findall(srcCfg[index_6]))    
                    if AP50_list_6 != []:
                        AP50_list_9 = (pattern.findall(srcCfg[index_9]))    
                    if AP50_list_9 != []:
                        AP50_list_12 = (pattern.findall(srcCfg[index_12]))
                    AP50_list = AP50_list_3 + AP50_list_6 + AP50_list_9 + AP50_list_12
                    AP50_list = [float(i) for i in AP50_list]
                    result[key] = AP50_list
                    result[f'm{key}'] = sum(AP50_list) / len(AP50_list)
        # if input('ck') == 'n':
        #     assert 1==2
        APClassWise = result['AP50']
        mAP = result['mAP50']
        ARClassWise = result['AR50']
        mAR = result['mAR50']
        # compute fscore
        FscoreClassWise = []
        for i in range(len(APClassWise)):
            FscoreClassWise.append(2*APClassWise[i]*ARClassWise[i]/(APClassWise[i]+ARClassWise[i]+0.00000001))
        mFscore = sum(FscoreClassWise)/len(FscoreClassWise)
        # print(len(APClassWise))

        for j in range(len(APClassWise), 5, 1):
            APClassWise.append(0.00)
            ARClassWise.append(0.00)
            FscoreClassWise.append(0.00)


    # read cT pT a version cfa_weight
    cT = float(re.findall(r"cT\d+\.?\d*", cfg_name)[0][2:])
    pT = float(re.findall(r"pT\d+\.?\d*", cfg_name)[0][2:])
    a = float(re.findall(r"a\d+\.?\d*", cfg_name)[0][1:])
    version = int(re.findall(r"-v\d+", cfg_name)[0][2:])
    cfa_weight = float(re.findall(r"9-\d+\.?\d*", cfg_name)[0][2:])
    try:
        DA_str = re.findall(r"DA-\d+-\d+-\d+-\d+", cfg_name)
        print(DA_str)
        DA_str = DA_str[0][3:]
        DA_str_list = DA_str.split('-')
    except:
        DA_str = re.findall(r"3class-[A-Za-z0-9]*", cfg_name)
        print(DA_str)
        DA_str = DA_str[0][7:]
        DA_str_list = [0,0,0,0]
    
        

    # write csv
    with open(csv_path, "a") as f:
        f.write("{}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {:.3f}, {:.3f}, {:.3f}, ".format(cfg_name, 
                                        DA_str, DA_str_list[0], DA_str_list[1], DA_str_list[2], DA_str_list[3], 
                                                              cT, pT, a, version, cfa_weight, mAP, mAR, mFscore))
        for i in range(5):
            f.write("{:.3f}, ".format(APClassWise[i]))
        for i in range(5):
            f.write("{:.3f}, ".format(ARClassWise[i]))
        for i in range(5):
            f.write("{:.3f}, ".format(FscoreClassWise[i]))
        f.write("\n")
    # save to list
    cfg_name_list.append(cfg_name)
    DA_list.append(DA_str)
    cT_list.append(cT)
    pT_list.append(pT)
    a_list.append(a)
    version_list.append(version)
    cfa_weight_list.append(cfa_weight)
    mAP_list.append(mAP)
    mAR_list.append(mAR)
    mFscore_list.append(mFscore)
    APClassWise_list.append(APClassWise)
    ARClassWise_list.append(ARClassWise)
    FscoreClassWise_list.append(FscoreClassWise)


# walk the path 
def WalkToFindLog(work_fdir, csv_path,  gpu=0):
    # walk work_fdir to find all folder
    for root, dirs, files in os.walk(work_fdir):
        is_py = False
        is_txt = False
        is_json = False
        is_log = False
        for file in files:
            if file.endswith('.py'):
                is_py = True
            if file.endswith('.txt'):
                is_txt = True
            if file.endswith('.json'):
                is_json = True
            if file.endswith('.log') and file.startswith('0_0th'):
                is_log = True
        if is_py and is_txt and is_json and is_log:
            work_dir = root
            # find config file
            file_list = files
            log_path = ''
            for file in file_list:
                if file.endswith('.log') and file.startswith('0_0th'):
                    log_path = os.path.join(work_dir, file)
                    break
            if log_path == '':
                print(root)
                assert 1==2
            read_log_save_csv(log_path, csv_path)
dir_name = 'CFA_ctpt'
dataset_name = 'HPT2HPL'
root_dir = os.getcwd().split('mmdet2')[0] + 'mmdet2'
work_fdir = os.path.join(f'{root_dir}/work_dirs/BIS/{dataset_name}',dir_name)
print(work_fdir)
csv_name = f'{dir_name}-aparf.csv'


csv_path = os.path.join(work_fdir, csv_name)

# add names to  csv, add each value of APClassWise, ARClassWise and FscoreClassWise to csv
with open(csv_path, "w") as f:    
    f.write("cfg_name, DA, DAb, DAn, DAp, DAo, cT, pT, a, version, cfa_weight, mAP, mAR, mFscore, ")
    for i in range(5):
        f.write("AP{}, ".format(i))
    for i in range(5):
        f.write("AR{}, ".format(i))
    for i in range(5):
        f.write("Fscore{}, ".format(i))
    f.write("\n")

WalkToFindLog(work_fdir, csv_path)

# compute average result
cT_set = list(set(cT_list))
cT_set.sort()
pT_set = list(set(pT_list))
pT_set.sort()
a_set = list(set(a_list))
a_set.sort()
DA_set = set(DA_list)
DA_set = list(DA_set)
DA_set.sort()
cfa_weight_set = set(cfa_weight_list)
for DA in DA_set:
    for cT in cT_set:
        for pT in pT_set:
            for a in a_set:
                for cfa_weight in cfa_weight_set:
                    # average ar ar fscore of the same DA cT pT a
                    mAP = []
                    mAR = []
                    mFscore = []
                    for i in range(len(cT_list)):
                        if DA_list[i] == DA and  cT_list[i] == cT and pT_list[i] == pT and a_list[i] == a and cfa_weight_list[i] == cfa_weight:
                            mAP.append(mAP_list[i])
                            mAR.append(mAR_list[i])
                            mFscore.append(mFscore_list[i])
                    if len(mAP) == 0:
                        continue
                    mAP = sum(mAP)/len(mAP)
                    mAR = sum(mAR)/len(mAR)
                    mFscore = sum(mFscore)/len(mFscore)
                    # write to csv
                    with open(csv_path, "a") as f:
                        version = 'a'
                        DA_str_list = DA.split('-')
                        if len(DA_str_list) <=1:
                            DA_str_list = [0,0,0,0]
                        f.write("{}, {}, ".format("", DA))
                        for i in range(len(DA_str_list)):
                            f.write("{}, ".format(DA_str_list[i]))
                        f.write("{}, {}, {}, {}, {}, {:.3f}, {:.3f}, {:.3f}, ".format(cT, pT, a, version, cfa_weight, mAP, mAR, mFscore))
                        f.write("\n")



/data/yebh/mmdet2/work_dirs/BIS/HPT2HPL/CFA_ctpt
/data/yebh/mmdet2/work_dirs/BIS/HPT2HPL/CFA_ctpt/yolov3-UDA-640-HPT2HPL-3class-DA-462-462-462-0-cfav9-0.075-cT0.4-pT0.5-a0.1-v1/0_0th_174_APARF.log
['', 'data', 'yebh', 'mmdet2', 'work_dirs', 'BIS', 'HPT2HPL', 'CFA_ctpt', 'yolov3-UDA-640-HPT2HPL-3class-DA-462-462-462-0-cfav9-0.075-cT0.4-pT0.5-a0.1-v1', '0_0th_174_APARF.log']
['DA-462-462-462-0']
/data/yebh/mmdet2/work_dirs/BIS/HPT2HPL/CFA_ctpt/yolov3-UDA-640-HPT2HPL-3class-DA-462-462-462-0-cfav9-0.075-cT0.5-pT0.4-a0.1-v1/0_0th_144_APARF.log
['', 'data', 'yebh', 'mmdet2', 'work_dirs', 'BIS', 'HPT2HPL', 'CFA_ctpt', 'yolov3-UDA-640-HPT2HPL-3class-DA-462-462-462-0-cfav9-0.075-cT0.5-pT0.4-a0.1-v1', '0_0th_144_APARF.log']
['DA-462-462-462-0']
/data/yebh/mmdet2/work_dirs/BIS/HPT2HPL/CFA_ctpt/yolov3-UDA-640-HPT2HPL-3class-DA-462-462-462-0-cfav9-0.075-cT0.6-pT0.5-a0.1-v1/0_0th_218_APARF.log
['', 'data', 'yebh', 'mmdet2', 'work_dirs', 'BIS', 'HPT2HPL', 'CFA_ctpt', 'yolov3-UDA-640-HPT2HPL-3class-DA